# Crawling dữ liệu OpenAQ

Notebook này hướng dẫn cách sử dụng OpenAQ API để tải dữ liệu đo chất lượng không khí (ví dụ PM2.5) và lưu thành CSV.

## 1. Thiết lập thư viện và cấu hình

Chúng ta dùng `requests` và `pandas`. Nếu chưa cài, hãy cài bằng `pip install requests pandas` trong môi trường ảo.

In [ ]:
import os
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

## 2. Hàm thu thập dữ liệu OpenAQ

Hàm sau gọi API `/v3/measurements`, quản lý phân trang và lưu dữ liệu về pandas DataFrame.

In [ ]:
OPENAQ_BASE_URL = 'https://api.openaq.org/v3'

def get_openaq_headers():
    api_key = os.environ.get('OPENAQ_API_KEY')
    return {'X-API-Key': api_key} if api_key else {}

def fetch_openaq_measurements(parameter='pm25', country=None, city=None, location=None, location_id=None,
                               start_date=None, end_date=None, limit=100, max_pages=20):
    url = f'{OPENAQ_BASE_URL}/measurements'
    all_results = []
    page = 1

    params = {
        'parameter': parameter,
        'limit': limit,
        'page': page,
        'sort': 'datetime'
    }

    if country is not None:
        params['country'] = country
    if city is not None:
        params['city'] = city
    if location is not None:
        params['location'] = location
    if location_id is not None:
        params['location_id'] = location_id
    if start_date is not None:
        params['date_from'] = start_date
    if end_date is not None:
        params['date_to'] = end_date

    headers = get_openaq_headers()

    while page <= max_pages:
        params['page'] = page
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        payload = response.json()

        results = payload.get('results', [])
        if not results:
            break

        all_results.extend(results)
        if len(results) < limit:
            break
        page += 1

    if not all_results:
        return pd.DataFrame()


    df = pd.json_normalize(all_results)   
    return df

## 2.1. Test với JSON mẫu

Dùng JSON mẫu để kiểm tra cách `pandas.json_normalize` chuyển đổi kết quả `results` thành DataFrame.

In [ ]:
sample_json = {
    "meta": {
        "name": "openaq-api",
        "website": "/",
        "page": 1,
        "limit": 100,
        "found": 1
    },
    "results": [
        {
            "id": 8118,
            "name": "New Delhi",
            "locality": "India",
            "timezone": "Asia/Kolkata",
            "country": {"id": 9, "code": "IN", "name": "India"},
            "owner": {"id": 4, "name": "Unknown Governmental Organization"},
            "provider": {"id": 119, "name": "AirNow"},
            "isMobile": False,
            "isMonitor": True,
            "instruments": [{"id": 2, "name": "Government Monitor"}],
            "sensors": [
                {
                    "id": 23534,
                    "name": "pm25 µg/m³",
                    "parameter": {
                        "id": 2,
                        "name": "pm25",
                        "units": "µg/m³",
                        "displayName": "PM2.5"
                    }
                }
            ],
            "coordinates": {"latitude": 28.63576, "longitude": 77.22445},
            "licenses": [
                {
                    "id": 33,
                    "name": "US Public Domain",
                    "attribution": {
                        "name": "Unknown Governmental Organization",
                        "url": None
                    },
                    "dateFrom": "2016-01-30",
                    "dateTo": None
                }
            ],
            "bounds": [77.22445, 28.63576, 77.22445, 28.63576],
            "distance": None,
            "datetimeFirst": {
                "utc": "2016-11-09T19:00:00Z",
                "local": "2016-11-10T00:30:00+05:30"
            },
            "datetimeLast": {
                "utc": "2024-12-13T14:30:00Z",
                "local": "2024-12-13T20:00:00+05:30"
            }
        }
    ]
}

class MockResponse:
    def __init__(self, json_data, status_code=200):
        self._json_data = json_data
        self.status_code = status_code

    def json(self):
        return self._json_data

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f"HTTP {self.status_code}")


def mock_requests_get(url, params=None, headers=None, timeout=None):
    return MockResponse(sample_json)

original_requests_get = requests.get
requests.get = mock_requests_get
try:
    df_sample = fetch_openaq_measurements(
        parameter='pm25',
        country='IN',
        start_date='2024-01-01T00:00:00Z',
        end_date='2024-01-02T00:00:00Z',
        limit=100,
        max_pages=1
    )
    print('Shape:', df_sample.shape)
    print(df_sample[['id', 'name', 'country.code', 'coordinates.latitude', 'coordinates.longitude']].to_dict(orient='records'))
    print('Sensors field type:', type(df_sample.loc[0, 'sensors']))
finally:
    requests.get = original_requests_get

## 2.2. Test với Location ID thực tế

Dùng `location_id=2161306` và khoảng thời gian 29/01/2024 - 30/01/2024.

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://api.openaq.org/v3"
OPENAQ_API_KEY = "8e8c3a875320048ecc88c70fcba8a72b10466adada7fb420c851038a59f3b987"   # <-- thay bằng key của bạn


def _normalize_parameter_name(x):
    return str(x).lower().replace(".", "").replace("_", "").replace(" ", "")


def get_location_metadata(location_id, api_key=OPENAQ_API_KEY):
    headers = {"X-API-Key": api_key}
    url = f"{BASE_URL}/locations/{location_id}"

    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()

    results = resp.json().get("results", [])
    if not results:
        raise ValueError(f"Không tìm thấy location_id={location_id}")

    loc = results[0]
    coords = loc.get("coordinates") or {}

    return {
        "location_id": loc.get("id"),
        "location_name": loc.get("name"),
        "locality": loc.get("locality"),
        "timezone": loc.get("timezone"),
        "latitude": coords.get("latitude"),
        "longitude": coords.get("longitude"),
    }

def get_sensor_id_from_location(location_id, parameter="pm25", api_key=OPENAQ_API_KEY):
    headers = {"X-API-Key": api_key}
    url = f"{BASE_URL}/locations/{location_id}/sensors"

    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()

    sensors = resp.json().get("results", [])
    target = _normalize_parameter_name(parameter)

    matched = [
        s for s in sensors
        if _normalize_parameter_name(s.get("parameter", {}).get("name", "")) == target
    ]

    if not matched:
        available = [s.get("parameter", {}).get("name", "") for s in sensors]
        raise ValueError(
            f"Không tìm thấy sensor cho parameter='{parameter}' tại location_id={location_id}. "
            f"Available: {available}"
        )

    # Lấy sensor đầu tiên khớp parameter
    return matched[0]["id"]

def fetch_openaq_measurements(
    parameter="pm25",
    sensor_id=None,
    location_id=None,
    start_date=None,
    end_date=None,
    limit=100,
    max_pages=10,
    api_key=OPENAQ_API_KEY
):
    if sensor_id is None and location_id is None:
        raise ValueError("Phải truyền sensor_id hoặc location_id.")

    headers = {"X-API-Key": api_key}

    # Nếu có location_id thì lấy metadata location trước
    location_meta = None
    if location_id is not None:
        location_meta = get_location_metadata(location_id, api_key=api_key)

    # Nếu chưa có sensor_id thì resolve từ location_id + parameter
    if sensor_id is None:
        sensor_id = get_sensor_id_from_location(
            location_id=location_id,
            parameter=parameter,
            api_key=api_key
        )

    all_rows = []
    page = 1

    while page <= max_pages:
        url = f"{BASE_URL}/sensors/{sensor_id}/measurements"
        params = {
            "datetime_from": start_date,
            "datetime_to": end_date,
            "limit": limit,
            "page": page
        }
        params = {k: v for k, v in params.items() if v is not None}

        resp = requests.get(url, headers=headers, params=params, timeout=60)
        resp.raise_for_status()

        payload = resp.json()
        results = payload.get("results", [])

        if not results:
            break

        for item in results:
            period = item.get("period") or {}
            dt_from = period.get("datetimeFrom") or {}
            dt_to = period.get("datetimeTo") or {}
            parameter_info = item.get("parameter") or {}

            all_rows.append({
                "location_id": location_id,
                "location_name": None if location_meta is None else location_meta.get("location_name"),
                "locality": None if location_meta is None else location_meta.get("locality"),
                "timezone": None if location_meta is None else location_meta.get("timezone"),
                "sensor_id": sensor_id,
                "parameter": parameter_info.get("name"),
                "units": parameter_info.get("units"),
                "value": item.get("value"),
                "datetime_utc": dt_from.get("utc"),
                "datetime_local": dt_from.get("local"),
                "datetime_to_utc": dt_to.get("utc"),
                "datetime_to_local": dt_to.get("local"),
                "latitude": None if location_meta is None else location_meta.get("latitude"),
                "longitude": None if location_meta is None else location_meta.get("longitude"),
            })

        if len(results) < limit:
            break

        page += 1

    df = pd.DataFrame(all_rows)

    if not df.empty:
        for col in ["datetime_utc", "datetime_local", "datetime_to_utc", "datetime_to_local"]:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")

        if "datetime_utc" in df.columns:
            df = df.sort_values("datetime_utc").reset_index(drop=True)

    return df

In [ ]:
# Thử fetch dữ liệu cho location_id = 2161306
# Location: Minh Khai - Bắc Từ Liêm, timezone Asia/Bangkok

start_date = '2024-01-29T00:00:00+07:00'
end_date = '2024-01-31T00:00:00+07:00'

df_location_test = fetch_openaq_measurements(
    parameter='pm25',
    location_id=2161306,
    start_date=start_date,
    end_date=end_date,
    limit=100,
    max_pages=10
)

print('Shape:', df_location_test.shape)
display(df_location_test.head(10))

In [ ]:
loc_meta = get_location_metadata(2161306)
print(loc_meta)

# Test API 6 station

In [ ]:
# TEST 6 LOCATION_ID TỪ FILE Excel
# Cell này giả định bạn đã chạy các cell phía trên:
# - fetch_openaq_measurements(...)
# - get_location_metadata(...)

import pandas as pd

locations_file = "D:/Bussiness_plan/Multimodal_PM25/data/raw/DataSample/Hanoi/Location_date.xlsx"   # đổi đường dẫn nếu cần
df_locations = pd.read_excel(locations_file)

# Chuẩn hoá tên cột đề phòng khác format
df_locations.columns = [str(c).strip() for c in df_locations.columns]

required_cols = {"Location_id", "Location_name", "Report Date"}
missing = required_cols - set(df_locations.columns)
if missing:
    raise ValueError(f"Thiếu cột trong file Excel: {missing}")

results = []

for _, row in df_locations.iterrows():
    location_id = int(row["Location_id"])
    requested_name = str(row["Location_name"]).strip()
    report_date = pd.to_datetime(row["Report Date"], dayfirst=True, errors="coerce")

    # Lấy cửa sổ test 2 ngày quanh Report Date
    if pd.isna(report_date):
        start_date = None
        end_date = None
    else:
        start_date = report_date.strftime("%Y-%m-%dT00:00:00+07:00")
        end_date = (report_date + pd.Timedelta(days=2)).strftime("%Y-%m-%dT00:00:00+07:00")

    item = {
        "location_id": location_id,
        "requested_name": requested_name,
        "report_date": report_date.date().isoformat() if pd.notna(report_date) else None,
        "metadata_ok": False,
        "measurement_ok": False,
        "api_location_name": None,
        "timezone": None,
        "latitude": None,
        "longitude": None,
        "pm25_rows": 0,
        "first_datetime_utc": None,
        "last_datetime_utc": None,
        "error": None,
    }

    try:
        # 1) Test metadata location_id có tồn tại không
        meta = get_location_metadata(location_id)
        item["metadata_ok"] = True
        item["api_location_name"] = meta.get("location_name")
        item["timezone"] = meta.get("timezone")
        item["latitude"] = meta.get("latitude")
        item["longitude"] = meta.get("longitude")

        # 2) Test fetch measurement PM2.5 quanh ngày mốc
        if start_date and end_date:
            df_test = fetch_openaq_measurements(
                parameter="pm25",
                location_id=location_id,
                start_date=start_date,
                end_date=end_date,
                limit=100,
                max_pages=5
            )

            item["pm25_rows"] = len(df_test)
            item["measurement_ok"] = len(df_test) > 0

            if not df_test.empty:
                # tìm cột thời gian phù hợp nếu có
                for col in ["datetime_utc", "datetime_local", "time_utc", "time_local", "datetime"]:
                    if col in df_test.columns:
                        item["first_datetime_utc"] = str(df_test[col].min())
                        item["last_datetime_utc"] = str(df_test[col].max())
                        break

    except Exception as e:
        item["error"] = str(e)

    results.append(item)

df_check = pd.DataFrame(results)

print("=== KẾT QUẢ TEST 6 LOCATION_ID ===")
display(df_check)

print("\n=== LOCATION_ID HỢP LỆ NHƯNG CHƯA CÓ PM2.5 TRONG WINDOW TEST ===")
display(df_check[(df_check["metadata_ok"] == True) & (df_check["measurement_ok"] == False)])

print("\n=== LOCATION_ID BỊ LỖI / KHÔNG NHẬN ===")
display(df_check[df_check["metadata_ok"] == False])

## 4. Lưu dữ liệu ra CSV

Lưu kết quả về thư mục `data/raw/openaq` để tái sử dụng.

## 5. Mở rộng

- Thêm lọc `city` hoặc `location` để tải dữ liệu cụ thể cho thành phố.
- Dùng `parameter='no2'`, `parameter='so2'`, `parameter='o3'` khi cần.
- Tăng `max_pages` hoặc điều chỉnh `start_date/end_date` cho dữ liệu dài hơn.

1. cấu hình cho 1 station

In [ ]:
import os
import time
import random
from pathlib import Path

import pandas as pd
import requests

OPENAQ_BASE_URL = "https://api.openaq.org/v3"
OPENAQ_API_KEY = OPENAQ_API_KEY = "8e8c3a875320048ecc88c70fcba8a72b10466adada7fb420c851038a59f3b987"   # <-- thay bằng key của bạn
if not OPENAQ_API_KEY:
    raise ValueError("Chưa có OPENAQ_API_KEY. Hãy set biến môi trường trước.")

HEADERS = {"X-API-Key": OPENAQ_API_KEY}

# =========================
# CHỈNH 1 STATION Ở ĐÂY
# =========================
LOCATION_ID = 2161307
   # <-- đổi sang station bạn muốn crawl

# Timeframe bạn đã chốt
START_DATE = "2024-01-29T00:00:00Z"
END_DATE   = "2026-04-20T23:59:59Z"

# 5 parameter mục tiêu
TARGET_PARAMETERS = {
    "co":   "CO mass µg/m³",
    "no2":  "NO₂ mass µg/m³",
    "o3":   "O₃ mass µg/m³",
    "pm25": "PM2.5 µg/m³",
    "so2":  "SO₂ mass µg/m³",
}

# chỉ crawl CSV
OUTPUT_DIR = Path("data/raw/one_station")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LIMIT = 1000
WINDOW_DAYS = 35

2. helper functions

In [ ]:
def request_json(url, params=None, timeout=60, max_retries=6):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=timeout)

            if resp.status_code == 429:
                reset_s = float(resp.headers.get("x-ratelimit-reset", 60))
                sleep_s = reset_s + random.uniform(1, 3)
                print(f"429 -> ngủ {sleep_s:.1f}s")
                time.sleep(sleep_s)
                continue

            if resp.status_code in {408, 500, 502, 503, 504}:
                sleep_s = min(60, 2 ** attempt + random.uniform(0.5, 1.5))
                print(f"{resp.status_code} -> retry sau {sleep_s:.1f}s")
                time.sleep(sleep_s)
                continue

            resp.raise_for_status()
            return resp.json()

        except requests.RequestException as e:
            if attempt == max_retries - 1:
                raise
            sleep_s = min(60, 2 ** attempt + random.uniform(0.5, 1.5))
            print(f"Lỗi mạng: {e} -> retry sau {sleep_s:.1f}s")
            time.sleep(sleep_s)


def get_location_metadata(location_id: int):
    payload = request_json(f"{OPENAQ_BASE_URL}/locations/{location_id}")
    results = payload.get("results", [])
    if not results:
        raise ValueError(f"Không tìm thấy location_id={location_id}")
    loc = results[0]
    coords = loc.get("coordinates") or {}
    return {
        "location_id": loc.get("id"),
        "location_name": loc.get("name"),
        "timezone": loc.get("timezone"),
        "latitude": coords.get("latitude"),
        "longitude": coords.get("longitude"),
    }


def get_location_sensors(location_id: int):
    all_rows = []
    page = 1

    while True:
        payload = request_json(
            f"{OPENAQ_BASE_URL}/locations/{location_id}/sensors",
            params={"limit": 1000, "page": page}
        )

        rows = payload.get("results", [])
        all_rows.extend(rows)

        meta = payload.get("meta", {})
        found = int(meta.get("found") or 0)

        if not rows or page * 1000 >= found:
            break
        page += 1

    return all_rows


def build_sensor_inventory_one_station(location_id: int):
    sensors = get_location_sensors(location_id)
    rows = []

    for s in sensors:
        p = s.get("parameter") or {}
        pname = str(p.get("name", "")).lower()
        units = p.get("units")

        if pname not in TARGET_PARAMETERS:
            continue

        # chỉ giữ đúng mass concentration theo µg/m³
        if units != "µg/m³":
            continue

        dt_first = s.get("datetimeFirst") or {}
        dt_last = s.get("datetimeLast") or {}

        rows.append({
            "location_id": location_id,
            "sensor_id": s.get("id"),
            "sensor_name": s.get("name"),
            "parameter_name": pname,
            "units": units,
            "datetimeFirst_utc": dt_first.get("utc"),
            "datetimeLast_utc": dt_last.get("utc"),
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["parameter_name", "datetimeFirst_utc", "sensor_id"]).reset_index(drop=True)
    return df


def split_windows(start_ts, end_ts, window_days=35):
    start_ts = pd.Timestamp(start_ts)
    end_ts = pd.Timestamp(end_ts)

    windows = []
    cur = start_ts
    step = pd.Timedelta(days=window_days)

    while cur <= end_ts:
        nxt = min(cur + step, end_ts)
        windows.append((cur, nxt))
        cur = nxt + pd.Timedelta(seconds=1)

    return windows


def fetch_sensor_hours(sensor_id: int, start_ts, end_ts):
    start_str = pd.Timestamp(start_ts).strftime("%Y-%m-%dT%H:%M:%SZ")
    end_str = pd.Timestamp(end_ts).strftime("%Y-%m-%dT%H:%M:%SZ")

    all_results = []
    page = 1

    while True:
        payload = request_json(
            f"{OPENAQ_BASE_URL}/sensors/{sensor_id}/hours",
            params={
                "datetime_from": start_str,
                "datetime_to": end_str,
                "limit": LIMIT,
                "page": page
            }
        )

        results = payload.get("results", [])
        if not results:
            break

        all_results.extend(results)

        meta = payload.get("meta", {})
        found = int(meta.get("found") or 0)

        if page * LIMIT >= found:
            break
        page += 1

    return all_results


def extract_simple_rows(results, parameter_name):
    rows = []

    for r in results:
        period = r.get("period") or {}
        dt_to = period.get("datetimeTo") or {}
        summary = r.get("summary") or {}

        rows.append({
            "datetime_utc": dt_to.get("utc"),
            "parameter_name": parameter_name,
            "value": summary.get("avg"),   # hourly mean
        })

    return rows

3. crawl 1 station và xuất CSV gọn

In [ ]:
meta = get_location_metadata(LOCATION_ID)
print("Station metadata:")
print(meta)

inventory = build_sensor_inventory_one_station(LOCATION_ID)
display(inventory)

if inventory.empty:
    raise ValueError("Station này không có sensor phù hợp với 5 parameter mass µg/m³ bạn yêu cầu.")

all_rows = []

global_start = pd.Timestamp(START_DATE)
global_end = pd.Timestamp(END_DATE)

for row in inventory.itertuples(index=False):
    sensor_start = pd.Timestamp(row.datetimeFirst_utc)
    sensor_end = pd.Timestamp(row.datetimeLast_utc)

    start_ts = max(global_start, sensor_start)
    end_ts = min(global_end, sensor_end)

    if start_ts > end_ts:
        continue

    windows = split_windows(start_ts, end_ts, window_days=WINDOW_DAYS)

    print(f"\n=== {row.parameter_name} | sensor_id={row.sensor_id} | {len(windows)} windows ===")

    for i, (w_start, w_end) in enumerate(windows, start=1):
        print(f"  Window {i}/{len(windows)}: {w_start} -> {w_end}")

        results = fetch_sensor_hours(row.sensor_id, w_start, w_end)
        rows = extract_simple_rows(results, row.parameter_name)
        all_rows.extend(rows)

simple_df = pd.DataFrame(all_rows)

if simple_df.empty:
    raise ValueError("Không lấy được hourly data cho station này trong timeframe đã chọn.")

# Chuẩn hóa
simple_df["datetime_utc"] = pd.to_datetime(simple_df["datetime_utc"], utc=True, errors="coerce")
simple_df = simple_df.dropna(subset=["datetime_utc"])
simple_df["parameter_name"] = simple_df["parameter_name"].str.lower()

# Nếu nhiều sensor cùng parameter chồng lên nhau, lấy mean theo giờ
simple_df = (
    simple_df
    .groupby(["datetime_utc", "parameter_name"], as_index=False)["value"]
    .mean()
)

# Pivot wide: 1 dòng = 1 giờ
final_df = simple_df.pivot(index="datetime_utc", columns="parameter_name", values="value").reset_index()

# Đổi tên cột theo yêu cầu
final_df = final_df.rename(columns=TARGET_PARAMETERS)

# Thêm thông tin vị trí station
final_df["location_id"] = meta["location_id"]
final_df["location_name"] = meta["location_name"]
final_df["latitude"] = meta["latitude"]
final_df["longitude"] = meta["longitude"]

# Giữ đúng thứ tự cột
final_columns = [
    "datetime_utc",
    "location_id",
    "location_name",
    "latitude",
    "longitude",
    "CO mass µg/m³",
    "NO₂ mass µg/m³",
    "O₃ mass µg/m³",
    "PM2.5 µg/m³",
    "SO₂ mass µg/m³",
]

for c in final_columns:
    if c not in final_df.columns:
        final_df[c] = pd.NA

final_df = final_df[final_columns].sort_values("datetime_utc").reset_index(drop=True)

# Lưu CSV
safe_name = str(meta["location_name"]).replace("/", "_").replace("\\", "_").strip()
output_csv = OUTPUT_DIR / f"{LOCATION_ID}_{safe_name}_5params_hourly_latlon.csv"

final_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print("\nĐã lưu CSV:")
print(output_csv.resolve())
print("Shape:", final_df.shape)
display(final_df.head(20))

# file PM2.5 hourly sang file modeling features

In [3]:
import pandas as pd
from pathlib import Path

# =========================
# CONFIG
# =========================
INPUT_CSV = Path(r"D:/Bussiness_plan/Multimodal_PM25/data/raw/DataSample/Hanoi/2161292_LuuQuangVu.csv")  # sửa path
OUTPUT_DIR = Path("D:/Bussiness_plan/Multimodal_PM25/data/raw/DataSample/Hanoi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PM25_COL = "PM2.5 µg/m³"   # đổi nếu file của bạn đang dùng tên khác
TIME_COL = "datetime_utc"

# =========================
# LOAD
# =========================
df = pd.read_csv(INPUT_CSV)

required_cols = [TIME_COL, PM25_COL]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Thiếu cột bắt buộc: {missing}")

df[TIME_COL] = pd.to_datetime(df[TIME_COL], utc=True, errors="coerce")
df = df.dropna(subset=[TIME_COL]).sort_values(TIME_COL).reset_index(drop=True)

# giữ target chuẩn tên ngắn gọn
df["pm25"] = pd.to_numeric(df[PM25_COL], errors="coerce")

# =========================
# TEMPORAL FEATURES
# =========================
df["year"] = df[TIME_COL].dt.year
df["month"] = df[TIME_COL].dt.month
df["day"] = df[TIME_COL].dt.day
df["hour"] = df[TIME_COL].dt.hour
df["day_of_week"] = df[TIME_COL].dt.dayofweek  # Monday=0, Sunday=6
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["is_dry_season"] = df["month"].isin([11, 12, 1, 2, 3, 4]).astype(int)

# =========================
# LAG FEATURES (NO LEAKAGE)
# =========================
df["pm25_lag1"] = df["pm25"].shift(1)
df["pm25_lag24"] = df["pm25"].shift(24)
df["pm25_lag168"] = df["pm25"].shift(168)

past_pm25 = df["pm25"].shift(1)

df["pm25_rolling_3h"] = past_pm25.rolling(window=3, min_periods=3).mean()
df["pm25_rolling_24h"] = past_pm25.rolling(window=24, min_periods=24).mean()
df["pm25_rolling_7d"] = past_pm25.rolling(window=168, min_periods=168).mean()
df["pm25_rolling_24h_std"] = past_pm25.rolling(window=24, min_periods=24).std()

# =========================
# CHỌN CỘT CUỐI
# =========================
base_cols = [c for c in ["location_id", "location_name", "latitude", "longitude"] if c in df.columns]

final_cols = (
    [TIME_COL] +
    base_cols +
    [
        "pm25",
        "year", "month", "day", "hour", "day_of_week",
        "is_weekend", "is_dry_season",
        "pm25_lag1", "pm25_lag24", "pm25_lag168",
        "pm25_rolling_3h", "pm25_rolling_24h",
        "pm25_rolling_7d", "pm25_rolling_24h_std"
    ]
)

feature_df = df[final_cols].copy()

# nếu muốn chỉ giữ những dòng đủ lag/rolling
feature_df_strict = feature_df.dropna().reset_index(drop=True)

# =========================
# SAVE
# =========================
safe_stem = INPUT_CSV.stem
out_csv_full = OUTPUT_DIR / f"{safe_stem}_pm25_features_full.csv"
out_csv_strict = OUTPUT_DIR / f"{safe_stem}_pm25_features_strict.csv"

feature_df.to_csv(out_csv_full, index=False, encoding="utf-8-sig")
feature_df_strict.to_csv(out_csv_strict, index=False, encoding="utf-8-sig")

print("Đã lưu file full:")
print(out_csv_full.resolve())
print("Shape full:", feature_df.shape)

print("\nĐã lưu file strict (dropna):")
print(out_csv_strict.resolve())
print("Shape strict:", feature_df_strict.shape)

display(feature_df.head(10))
display(feature_df_strict.head(10))

Đã lưu file full:
D:\Bussiness_plan\Multimodal_PM25\data\raw\DataSample\Hanoi\2161292_LuuQuangVu_pm25_features_full.csv
Shape full: (16414, 20)

Đã lưu file strict (dropna):
D:\Bussiness_plan\Multimodal_PM25\data\raw\DataSample\Hanoi\2161292_LuuQuangVu_pm25_features_strict.csv
Shape strict: (12452, 20)


,datetime_utc,location_id,location_name,latitude,longitude,pm25,year,month,day,hour,day_of_week,is_weekend,is_dry_season,pm25_lag1,pm25_lag24,pm25_lag168,pm25_rolling_3h,pm25_rolling_24h,pm25_rolling_7d,pm25_rolling_24h_std
0,2024-01-29 17:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,48.63,2024,1,29,17,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-29 18:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,52.74,2024,1,29,18,0,0,1,48.63,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-01-29 19:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,49.70,2024,1,29,19,0,0,1,52.74,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-01-29 20:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,53.02,2024,1,29,20,0,0,1,49.70,NaN,NaN,50.356667,NaN,NaN,NaN
4,2024-01-29 21:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,49.56,2024,1,29,21,0,0,1,53.02,NaN,NaN,51.820000,NaN,NaN,NaN
5,2024-01-29 22:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,45.92,2024,1,29,22,0,0,1,49.56,NaN,NaN,50.760000,NaN,NaN,NaN
6,2024-01-29 23:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,50.18,2024,1,29,23,0,0,1,45.92,NaN,NaN,49.500000,NaN,NaN,NaN
7,2024-01-30 00:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,50.59,2024,1,30,0,1,0,1,50.18,NaN,NaN,48.553333,NaN,NaN,NaN
8,2024-01-30 01:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,44.47,2024,1,30,1,1,0,1,50.59,NaN,NaN,48.896667,NaN,NaN,NaN
9,2024-01-30 02:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,45.57,2024,1,30,2,1,0,1,44.47,NaN,NaN,48.413333,NaN,NaN,NaN


,datetime_utc,location_id,location_name,latitude,longitude,pm25,year,month,day,hour,day_of_week,is_weekend,is_dry_season,pm25_lag1,pm25_lag24,pm25_lag168,pm25_rolling_3h,pm25_rolling_24h,pm25_rolling_7d,pm25_rolling_24h_std
0,2024-02-07 10:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,17.99,2024,2,7,10,2,0,1,10.54,12.05,48.63,6.290000,5.427500,18.521250,2.273485
1,2024-02-07 11:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,13.62,2024,2,7,11,2,0,1,17.99,8.88,52.74,11.020000,5.675000,18.338869,3.171679
2,2024-02-07 12:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,13.03,2024,2,7,12,2,0,1,13.62,6.73,49.70,14.050000,5.872500,18.106012,3.509518
3,2024-02-07 13:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,20.06,2024,2,7,13,2,0,1,13.03,6.33,53.02,14.880000,6.135000,17.887738,3.800030
4,2024-02-07 14:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,24.13,2024,2,7,14,2,0,1,20.06,5.93,49.56,15.570000,6.707083,17.691548,4.746340
5,2024-02-07 15:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,21.92,2024,2,7,15,2,0,1,24.13,4.91,45.92,19.073333,7.465417,17.540179,5.924491
6,2024-02-07 16:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,16.06,2024,2,7,16,2,0,1,21.92,4.10,50.18,22.036667,8.174167,17.397321,6.586018
7,2024-02-07 17:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,11.40,2024,2,7,17,2,0,1,16.06,5.32,50.59,20.703333,8.672500,17.194226,6.715547
8,2024-02-07 18:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,8.47,2024,2,7,18,2,0,1,11.40,6.16,44.47,16.460000,8.925833,16.960952,6.698237
9,2024-02-07 19:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,8.04,2024,2,7,19,2,0,1,8.47,5.98,45.57,11.976667,9.022083,16.746667,6.673316


# gộp 2 station và tạo 1 file chung

In [5]:
from pathlib import Path
import pandas as pd

# =========================
# CONFIG
# =========================
BASE_DIR = Path(r"D:/Bussiness_plan/Multimodal_PM25/data/raw/DataSample/Hanoi")   # đổi nếu cần

FILES = {
    "2161292": {
        "features": BASE_DIR / "2161292_LuuQuangVu_pm25_features_strict.csv",
        "pollutants": BASE_DIR / "2161292_LuuQuangVu.csv",
    },
    "2161306": {
        "features": BASE_DIR / "2161306_MinhKhai_pm25_features_strict.csv",
        "pollutants": BASE_DIR / "2161306_MinhKhai.csv",
    },
}

OUTPUT_DIR = BASE_DIR / "merged_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MERGE_KEYS = ["datetime_utc", "location_id"]

# chỉ lấy thêm 4 pollutant covariates
POLLUTANT_COLS = [
    "datetime_utc",
    "location_id",
    "CO mass µg/m³",
    "NO₂ mass µg/m³",
    "O₃ mass µg/m³",
    "SO₂ mass µg/m³",
]

RENAME_POLLUTANTS = {
    "CO mass µg/m³": "co_ugm3",
    "NO₂ mass µg/m³": "no2_ugm3",
    "O₃ mass µg/m³": "o3_ugm3",
    "SO₂ mass µg/m³": "so2_ugm3",
}

def load_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path.resolve()}")
    df = pd.read_csv(path)
    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True, errors="coerce")
    df = df.dropna(subset=["datetime_utc"]).copy()
    return df

def merge_one_station(feature_path, pollutant_path, station_tag):
    feat = load_csv(feature_path)
    pol = load_csv(pollutant_path)

    # Giữ đúng cột cần thiết từ pollutant file
    missing_pol = [c for c in POLLUTANT_COLS if c not in pol.columns]
    if missing_pol:
        raise ValueError(f"{station_tag}: thiếu cột trong pollutant file: {missing_pol}")

    pol = pol[POLLUTANT_COLS].copy()
    pol = pol.rename(columns=RENAME_POLLUTANTS)

    # Bỏ duplicate nếu có
    feat = feat.sort_values(MERGE_KEYS).drop_duplicates(subset=MERGE_KEYS, keep="last")
    pol = pol.sort_values(MERGE_KEYS).drop_duplicates(subset=MERGE_KEYS, keep="last")

    # Merge: dùng feature file làm base
    merged = feat.merge(
        pol,
        on=MERGE_KEYS,
        how="left",
        validate="one_to_one"
    )

    # Sắp xếp lại cột cho gọn
    preferred_cols = [
        "datetime_utc",
        "location_id",
        "location_name",
        "latitude",
        "longitude",
        "pm25",
        "co_ugm3",
        "no2_ugm3",
        "o3_ugm3",
        "so2_ugm3",
        "year",
        "month",
        "day",
        "hour",
        "day_of_week",
        "is_weekend",
        "is_dry_season",
        "pm25_lag1",
        "pm25_lag24",
        "pm25_lag168",
        "pm25_rolling_3h",
        "pm25_rolling_24h",
        "pm25_rolling_7d",
        "pm25_rolling_24h_std",
    ]

    for c in preferred_cols:
        if c not in merged.columns:
            merged[c] = pd.NA

    merged = merged[preferred_cols].sort_values("datetime_utc").reset_index(drop=True)

    out_path = OUTPUT_DIR / f"{station_tag}_merged.csv"
    merged.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"\nĐã lưu file merged cho {station_tag}:")
    print(out_path.resolve())
    print("Shape:", merged.shape)

    return merged

# =========================
# MERGE TỪNG STATION
# =========================
merged_frames = []

for station_tag, paths in FILES.items():
    merged_df = merge_one_station(
        feature_path=paths["features"],
        pollutant_path=paths["pollutants"],
        station_tag=station_tag
    )
    merged_frames.append(merged_df)

# =========================
# FILE CHUNG 2 STATION
# =========================
all_stations = pd.concat(merged_frames, ignore_index=True)
all_stations = all_stations.sort_values(["location_id", "datetime_utc"]).reset_index(drop=True)

all_path = OUTPUT_DIR / "all_stations_merged.csv"
all_stations.to_csv(all_path, index=False, encoding="utf-8-sig")

print("\nĐã lưu file chung:")
print(all_path.resolve())
print("Shape all:", all_stations.shape)

display(all_stations.head(10))


Đã lưu file merged cho 2161292:
D:\Bussiness_plan\Multimodal_PM25\data\raw\DataSample\Hanoi\merged_data\2161292_merged.csv
Shape: (12452, 24)

Đã lưu file merged cho 2161306:
D:\Bussiness_plan\Multimodal_PM25\data\raw\DataSample\Hanoi\merged_data\2161306_merged.csv
Shape: (3756, 24)

Đã lưu file chung:
D:\Bussiness_plan\Multimodal_PM25\data\raw\DataSample\Hanoi\merged_data\all_stations_merged.csv
Shape all: (16208, 24)


,datetime_utc,location_id,location_name,latitude,longitude,pm25,co_ugm3,no2_ugm3,o3_ugm3,so2_ugm3,...,day_of_week,is_weekend,is_dry_season,pm25_lag1,pm25_lag24,pm25_lag168,pm25_rolling_3h,pm25_rolling_24h,pm25_rolling_7d,pm25_rolling_24h_std
0,2024-02-07 10:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,17.99,879.6,NaN,26.84,2.39,...,2,0,1,10.54,12.05,48.63,6.290000,5.427500,18.521250,2.273485
1,2024-02-07 11:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,13.62,745.3,NaN,27.02,2.40,...,2,0,1,17.99,8.88,52.74,11.020000,5.675000,18.338869,3.171679
2,2024-02-07 12:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,13.03,655.1,NaN,24.59,2.36,...,2,0,1,13.62,6.73,49.70,14.050000,5.872500,18.106012,3.509518
3,2024-02-07 13:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,20.06,628.8,NaN,28.76,2.44,...,2,0,1,13.03,6.33,53.02,14.880000,6.135000,17.887738,3.800030
4,2024-02-07 14:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,24.13,560.2,NaN,23.59,2.40,...,2,0,1,20.06,5.93,49.56,15.570000,6.707083,17.691548,4.746340
5,2024-02-07 15:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,21.92,548.7,NaN,26.58,2.40,...,2,0,1,24.13,4.91,45.92,19.073333,7.465417,17.540179,5.924491
6,2024-02-07 16:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,16.06,496.6,NaN,25.43,2.30,...,2,0,1,21.92,4.10,50.18,22.036667,8.174167,17.397321,6.586018
7,2024-02-07 17:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,11.40,489.1,NaN,28.73,2.43,...,2,0,1,16.06,5.32,50.59,20.703333,8.672500,17.194226,6.715547
8,2024-02-07 18:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,8.47,492.3,NaN,26.34,2.30,...,2,0,1,11.40,6.16,44.47,16.460000,8.925833,16.960952,6.698237
9,2024-02-07 19:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,8.04,469.8,NaN,28.72,2.37,...,2,0,1,8.47,5.98,45.57,11.976667,9.022083,16.746667,6.673316
